# Chạy web app trên Kaggle GPU

Chạy BE thật (worker + TotalSpineSeg trên GPU) và FE trên Kaggle, mở web qua tunnel Cloudflare.

1. Kaggle: *Settings → Accelerator → GPU T4 x2* và *Settings → Internet → On* (tài khoản phải xác minh số điện thoại mới bật được Internet). Rồi *Run All*.
2. Ô đầu `git clone` repo (lần sau thì `git pull`) theo `BRANCH`, rồi build FE (`fe/dist` không có trong git).
3. Mở link `trycloudflare.com` ở ô tunnel, upload zip DICOM hoặc `.nii.gz` như bình thường.

Kaggle chỉ thấy code **đã push** lên GitHub. Ô bật BE tải khoảng 4 GB weights ở lần chạy đầu (vài phút); sau đó mỗi job khoảng vài chục giây.
Kaggle không có proxy cổng như Colab nên phải dùng tunnel: **URL `trycloudflare.com` là public**, ai có link đều vào được và app không có đăng nhập, nên chỉ dùng với dữ liệu demo, tắt tunnel (`tunnel.terminate()`) khi xong.
Session tắt thì mất BE và tunnel; giữ tab notebook mở trong lúc dùng web.

In [ ]:
import os, re, subprocess, sys, time, urllib.request
from pathlib import Path

REPO_URL = "https://github.com/hieunghuu/mmis-gr8-lowbackpain"
BRANCH = "main"
WORK = Path("/kaggle/working")
ROOT = WORK / "mmis-gr8-lowbackpain"
DATA = WORK / "data"
NODE = "v22.20.0"   # fe/package.json cần Node >= 22.18

if (ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "pull", "-q", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

node_dir = WORK / f"node-{NODE}-linux-x64"
if not node_dir.exists():
    subprocess.run(f"curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C {WORK}",
                   shell=True, check=True)
os.environ["PATH"] = f"{node_dir}/bin:" + os.environ["PATH"]
subprocess.run("npm ci --no-audit --no-fund && npm run build", shell=True, cwd=ROOT / "fe", check=True)
assert (ROOT / "fe" / "dist" / "index.html").exists()

In [ ]:
%pip install -q -r {ROOT}/be/requirements.txt -r {ROOT}/pipeline/requirements.txt "totalspineseg[nnunetv2]" "kornia<0.8"

import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"
# torch mới bỏ hỗ trợ P100 (sm_60): is_available() vẫn True nhưng chạy kernel thì lỗi -> thử 1 phép tính
torch.ones(1, device="cuda").add_(1)
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# chạy lại ô này thì dừng BE cũ trước, tránh 2 worker tranh GPU
if "proc" in globals() and proc.poll() is None:
    proc.terminate(); proc.wait()

env = {**os.environ, "DATA_DIR": str(DATA), "TOTALSPINESEG_DATA": str(DATA / "models"),
       "SERVE_FRONTEND": "1"}

# Lần đầu: nạp DB + job demo đã commit ở demo-official/data, để web có sẵn kết quả xem ngay.
# DATA đã có app.db (chạy lại ô này / job mới) thì không ghi đè.
import shutil
SEED = ROOT / "demo-official" / "data"
if not (DATA / "app.db").exists() and (SEED / "app.db").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SEED / "app.db", DATA / "app.db")
    shutil.copytree(SEED / "jobs", DATA / "jobs", dirs_exist_ok=True)
    print("Đã nạp job demo:", len(list((DATA / "jobs").iterdir())))

# Tải weights (~4 GB) trước khi bật BE, để job đầu không phải tải trong giới hạn JOB_TIMEOUT_SEC.
# TotalSpineSeg thấy file zip đã có thì không tải lại: zip dở dang (lần tải trước bị ngắt) phải xóa trước.
import zipfile
for z in (DATA / "models/nnUNet/exports").glob("*.zip"):
    if not zipfile.is_zipfile(z):
        print("Xóa zip hỏng:", z.name); z.unlink()
subprocess.run(["totalspineseg_init"], env=env, check=True)
be_log = WORK / "be.log"
proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "be.app.main:app", "--host", "127.0.0.1", "--port", "8000"],
                        cwd=ROOT, env=env, stdout=open(be_log, "a"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8000/api/v1/health").read().decode())
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError(be_log.read_text()[-3000:])
for path in ["/api/v1/health", "/"]:
    print(path, urllib.request.urlopen("http://127.0.0.1:8000" + path).status)

### Mở web qua tunnel
**URL in ra là public**: không upload dữ liệu bệnh nhân thật. Xong thì chạy `tunnel.terminate()`.

In [ ]:
cf = WORK / "cloudflared"
if not cf.exists():
    subprocess.run(["curl", "-fsSL", "-o", cf,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    cf.chmod(0o755)
if "tunnel" in globals() and tunnel.poll() is None:
    tunnel.terminate(); tunnel.wait()
cf_log = WORK / "cloudflared.log"
tunnel = subprocess.Popen([cf, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.DEVNULL, stderr=open(cf_log, "w"))
for _ in range(30):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", cf_log.read_text())
    if m:
        print("Mở web:", m.group())
        break
    time.sleep(1)
else:
    raise RuntimeError(cf_log.read_text()[-2000:])

### Log
Log BE và log TotalSpineSeg của job mới nhất.

In [ ]:
!tail -n 30 {be_log}
logs = sorted((DATA / "jobs").glob("*/logs/totalspineseg.log"), key=os.path.getmtime)
if logs:
    print("\n==", logs[-1]); print(logs[-1].read_text()[-3000:])

### Tải DB, job và log về máy
Nén `app.db` + `data/jobs` (ảnh input, mask, log từng job) + `be.log` + `cloudflared.log` thành `export.zip`, bỏ `models` (~4 GB).
Bấm link in ra, hoặc mở panel bên phải *Output → /kaggle/working → export.zip → Download*.

In [ ]:
import sqlite3, zipfile
from IPython.display import FileLink

# DB đang ở chế độ WAL: copy thẳng file có thể thiếu các ghi gần nhất -> dùng backup API của sqlite
snap = WORK / "app_snapshot.db"
with sqlite3.connect(DATA / "app.db") as src, sqlite3.connect(snap) as dst:
    src.backup(dst)

out = WORK / "export.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(snap, "data/app.db")
    for f in (DATA / "jobs").rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(WORK))
    for f in [be_log, WORK / "cloudflared.log"]:
        if f.exists():
            zf.write(f, f.name)
snap.unlink()
print(f"{out} ({out.stat().st_size / 1e6:.1f} MB)")
FileLink(os.path.relpath(out))   # FileLink cần đường dẫn tương đối với thư mục chạy notebook